# EXP-008 — ByT5 Joint L1+L2+L3 Classification (Kaggle T4)

**Purpose:** Fine-tune `google/byt5-base` on the ArabicITSM-9K dataset  
using the identical three-task (L1+L2+L3) configuration as EXP-006a (MARBERTv2).  
The only variable that changes is the pretrained encoder — all hyperparameters,  
data splits, preprocessing, and architecture are identical to EXP-006a.

**Comparison target:** `kaggle_train_l1l2l3_arabic_itsm_classification.ipynb` (EXP-006a)  
**Model:** `google/byt5-base` (~582M params, byte-level T5 encoder, token-free)  
**Tasks:** L1 (6 classes) + L2 (16 classes) + L3 (48 classes)  
**Hardware:** Kaggle Tesla T4 (16 GB VRAM)  
**Output:** `models/byt5_l1_l2_l3_best/`

**Key difference from BERT-family:** ByT5 uses byte-level tokenization (no vocabulary).  
The classifier uses mean pooling over encoder outputs instead of the [CLS] token.  
Max sequence length is set to 512 bytes to cover the full dataset without truncation.

In [ ]:
# Clone the repository and install dependencies
# Mirrors the setup cell in kaggle_train_l1l2l3_arabic_itsm_classification.ipynb exactly
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding Kaggle's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml -q

In [ ]:
# Copy processed data from Kaggle dataset input
# Mirrors kaggle_train_l1l2l3_arabic_itsm_classification.ipynb exactly
!rm -rf data/processed && mkdir -p data/processed
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/

!ls data/processed
# Expected: label_encoders.pkl  test.csv  train.csv  val.csv

In [ ]:
!nvidia-smi

In [ ]:
# Download ByT5-base fully to local disk before training.
# This replaces HuggingFace lazy-streaming with a single complete download.
from huggingface_hub import snapshot_download

print('Downloading google/byt5-base to local disk...')
BYT5_PATH = snapshot_download(
    'google/byt5-base',
    local_dir='/kaggle/working/model_cache/byt5'
)
print('ByT5-base ready at:', BYT5_PATH)

In [ ]:
# EXP-008: ByT5 joint L1+L2+L3 training
#
# --batch-size 4:   ByT5-base OOMs at batch=16 with long sequences on a 16GB T4.
# --max-length 256: Covers full corpus (~200-300 bytes per ticket).
# --no-fp16:        ByT5 is unstable with FP16 (gradient overflow causes frozen F1
#                   across all epochs). ByT5 must be trained in FP32.
# --lr 1e-5:        Same as EXP-006a (MARBERTv2).
#
# The classifier auto-detects the T5 family and uses T5EncoderModel + mean pooling.
# Note: train.py names the output marbert_{task_label}_best — renamed in next cell.

!python scripts/train.py \
    --model /kaggle/working/model_cache/byt5 \
    --tasks l1 l2 l3 \
    --data-dir data/processed \
    --epochs 10 \
    --lr 1e-5 \
    --batch-size 4 \
    --max-length 256 \
    --no-fp16

In [ ]:
import os

# train.py always names output marbert_{task_label}_best — rename to byt5_ for clarity
src = 'models/marbert_l1_l2_l3_best'
dst = 'models/byt5_l1_l2_l3_best'

if os.path.exists(src) and not os.path.exists(dst):
    os.rename(src, dst)
    print(f'Renamed: {src} → {dst}')
elif os.path.exists(dst):
    print(f'Already exists: {dst}')
else:
    raise FileNotFoundError(f'Expected checkpoint not found: {src}')

In [ ]:
import os, shutil, torch

checkpoint_dir = 'models/byt5_l1_l2_l3_best'
print('Checkpoint files:', os.listdir(checkpoint_dir))

# Verify heads.pt contains exactly the 3 expected task heads
heads = torch.load(f'{checkpoint_dir}/heads.pt', map_location='cpu', weights_only=False)
print('Head keys:', sorted(heads.keys()))

expected_keys = {'l1.1.weight', 'l1.1.bias', 'l2.1.weight', 'l2.1.bias', 'l3.1.weight', 'l3.1.bias'}
assert expected_keys == set(heads.keys()), f'Unexpected keys: {set(heads.keys())}'
print('\nVerification passed: 3 joint heads found (l1, l2, l3)')

# Print test metrics
import json
metrics_path = f'{checkpoint_dir}/test_metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('\nTest metrics:')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')

# Create download archive
shutil.make_archive('/kaggle/working/byt5_l1_l2_l3_best', 'zip', 'models', 'byt5_l1_l2_l3_best')
print('\nDownload archive: /kaggle/working/byt5_l1_l2_l3_best.zip')
print('Place in: arabic-itsm-classification/models/byt5_l1_l2_l3_best/')

## Next Steps

1. Download `byt5_l1_l2_l3_best.zip` from Kaggle output
2. Extract to `arabic-itsm-classification/models/byt5_l1_l2_l3_best/`
3. Record L1/L2/L3 macro-F1 results from the test metrics printed above
4. Fill in Section 7 of the paper (`section-07-arabert-comparison.md`) with ByT5 results

**Expected comparison (hypothesis — fill in after run):**

| Head | MARBERTv2 EXP-006a | ByT5 EXP-008 |
|------|:------------------:|:------------:|
| L1 Macro-F1 | 88.38% | TBD |
| L2 Macro-F1 | 87.05% | TBD |
| L3 Macro-F1 | 77.52% | TBD |

Hypothesis: MARBERTv2 outperforms ByT5 at coarser levels (L1/L2) due to dialect-specific  
pretraining; ByT5 may recover at L3 due to byte-level robustness to spelling variation.